# 03 · Optuna: hiperparâmetros do Transformer (alvo retorno)

Mesmo protocolo da etapa 02, agora com o Transformer.
- Busca só nos dias antes do teste: validação cruzada temporal em 5 blocos.
- Critério: MAPE da rede ÷ MAPE do naive (< 1 = melhor que o naive), média das moedas.
- Teste com 5 sementes: Transformer padrão x Transformer Optuna x naive, e a LSTM Optuna da etapa 02 ao lado.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import pandas as pd, matplotlib.pyplot as plt
import optuna
from tcc import avaliacao, config, experimento, otimizacao

In [ ]:
ALVO   = 'retorno'
INICIO = '2022-01-01'
FIM    = '2023-12-31'
TRIALS = config.TRIALS_OPTUNA
MOEDAS = config.MOEDAS

ETAPA_02 = os.path.join(RAIZ, 'results', '02_lstm', 'previsoes.csv')
SAIDA = config.PASTA_RESULTADOS / '03_transformer'
SAIDA.mkdir(parents=True, exist_ok=True)

## Busca

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
estudo, HP = otimizacao.buscar(ALVO, INICIO, FIM, trials=TRIALS, moedas=MOEDAS, arquitetura='Transformer')
trials = estudo.trials_dataframe().sort_values('value')
trials.to_csv(SAIDA / 'trials.csv', index=False)
(SAIDA / 'hiperparametros.json').write_text(json.dumps(HP, indent=2))
print('Melhor erro relativo ao naive:', round(estudo.best_value, 4)); print(HP)
trials.head(5).round(4)

In [ ]:
ordem = trials.sort_values('number')
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(ordem['number'], ordem['value'], 'o', alpha=.6, label='trial')
ax.plot(ordem['number'], ordem['value'].cummin(), 'k-', label='melhor até aqui')
ax.axhline(1, color='r', ls='--', label='naive')
ax.set_xlabel('trial'); ax.set_ylabel('MAPE rede ÷ MAPE naive'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(SAIDA / 'optuna.png', dpi=130); plt.show()

## Teste: Transformer padrão x Transformer Optuna (e LSTM Optuna da etapa 02)

In [ ]:
modelos = {'Transformer padrão': config.HP_TRANSFORMER, 'Transformer Optuna': HP}
prev = pd.concat([experimento.rodar(m, ALVO, INICIO, FIM, hp=hp).assign(modelo=nome)
                  for m in MOEDAS for nome, hp in modelos.items()], ignore_index=True)
lstm = pd.read_csv(ETAPA_02, parse_dates=['data']).query("modelo == 'LSTM Optuna' and moeda in @MOEDAS")
prev = pd.concat([prev, lstm], ignore_index=True)
prev.to_csv(SAIDA / 'previsoes.csv', index=False)
resumo = experimento.resumir(prev, chave='modelo')
resumo.round(4).to_csv(SAIDA / 'resumo.csv', index=False)
resumo.round(3)

In [ ]:
colunas = ['LSTM Optuna', *modelos]
tabela = resumo.pivot(index='moeda', columns='modelo', values='MAPE_%').loc[MOEDAS, colunas]
tabela.insert(0, 'naive', resumo.groupby('moeda')['MAPE_naive_%'].first())
tabela.loc['média'] = tabela.mean()
tabela.round(3)

## Diebold-Mariano: Transformer Optuna x LSTM Optuna

DM < 0: o Transformer erra menos.

In [ ]:
media = prev.groupby(['moeda', 'modelo', 'data'])[['real', 'previsto']].mean()
linhas = []
for m in MOEDAS:
    t, l = media.loc[(m, 'Transformer Optuna')], media.loc[(m, 'LSTM Optuna')]
    dm, p = avaliacao.diebold_mariano(t.previsto - t.real, l.previsto - l.real)
    linhas.append({'moeda': m, 'DM': dm, 'p': p})
dm_modelos = pd.DataFrame(linhas)
dm_modelos.round(4).to_csv(SAIDA / 'diebold_mariano.csv', index=False)
dm_modelos.round(3)

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'alvo': ALVO, 'inicio': INICIO, 'fim': FIM, 'trials': TRIALS,
    'dobras_cv': config.DOBRAS_CV, 'moedas': MOEDAS, 'sementes': config.SEMENTES,
    'hp_padrao': config.HP_TRANSFORMER, 'hp_optuna': HP}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))